# ch05 Bonus 13：Gemma4 实现

> 对照官方 `ch05/17_gemma4`（官方目录编号）
> **参考真实模型**：Google Gemma 系列（2025 最新）

## 一句话

Gemma4 在 Gemma3 基础上引入 **更深层的滑动窗口比例、muP（最大更新参数化）** 等改进，进一步优化超参迁移。

## 相对 Gemma3 的演进

- 滑动窗口的全局:局部比例调整（更多局部层省算力）
- muP（maximum update parameterization）：让小模型的超参能直接迁移到大模型，省去大模型调参
- 注意力机制的进一步微调

> 本 notebook 聚焦 muP 的核心思想演示。

In [ ]:
import torch
import torch.nn as nn

# muP 核心思想：让不同宽度模型的「每层更新幅度」保持一致
# 关键：参数初始化和 lr 随宽度缩放，而非固定

def mup_init(layer, fan_in, width_ratio=1.0):
    """muP 风格初始化：标准初始化按 √width 缩放，使更新幅度跨宽度一致。"""
    # 普通 init：std = 1/sqrt(fan_in)
    # muP：某些层用 std = 1/fan_in（而非 1/sqrt(fan_in)），配合 lr 调整
    std = 1.0 / (fan_in ** 0.5) / (width_ratio ** 0.5)
    nn.init.normal_(layer.weight, mean=0, std=std)

# 对比：不同宽度下，普通 init 的激活方差会漂移，muP 保持稳定
print("muP 思想：让小模型的训练动态（更新幅度）能迁移到大模型")
for width in [128, 512, 2048]:
    layer = nn.Linear(width, width, bias=False)
    mup_init(layer, width)
    x = torch.randn(1, 16, width)
    out = layer(x)
    print(f"  width={width:5d}: 输出方差 {out.var():.4f}（muP 旨在跨宽度保持稳定）")
print("\n💡 muP 让你在小模型上调好的 lr/初始化，直接用到大模型而不需重新搜索。")